# Tunisian License Plate Detection & Recognition - KAGGLE READY VERSION
**Author**: AI Assistant  
**Description**: End-to-end pipeline for Detection (VGG16) and Recognition (ResNet18) optimized for Kaggle Environment.

In [ ]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
import zipfile
import re

warnings.filterwarnings('ignore')

## 1. ENVIRONMENT & PATH CONFIGURATION

In [ ]:
def setup_paths():
    """
    Automatically detects paths for Kaggle, Colab, or Local environments.
    Returns a dictionary of paths.
    """
    # Check environment
    if os.path.exists('/kaggle/input'):
        env = 'kaggle'
        base_input = '/kaggle/input'
        base_working = '/kaggle/working'
    elif os.path.exists('/content'):
        env = 'colab'
        base_input = '/content' # In Colab, data is usually uploaded to /content or mounted drive
        base_working = '/content/working'
    else:
        env = 'local'
        base_input = './data' # Expect data in ./data folder locally
        base_working = './output'

    print(f"Detected Environment: {env.upper()}")

    # Specific logic for Kaggle to find the dataset folder name dynamically
    if env == 'kaggle':
        dataset_name = None
        for item in os.listdir(base_input):
            item_path = os.path.join(base_input, item)
            if os.path.isdir(item_path):
                # Look for keywords typical of the challenge
                if 'tunisia' in item.lower() or 'license' in item.lower() or 'plate' in item.lower():
                    dataset_name = item
                    break
        if not dataset_name:
            # Fallback: take the first directory if no keyword match
            dirs = [d for d in os.listdir(base_input) if os.path.isdir(os.path.join(base_input, d))]
            if dirs:
                dataset_name = dirs[0]
            else:
                raise FileNotFoundError("No dataset directory found in /kaggle/input")
        
        DATA_DIR = os.path.join(base_input, dataset_name)
        print(f"Found Dataset Directory: {DATA_DIR}")
    else:
        DATA_DIR = base_input

    # Define all paths
    paths = {
        'data_dir': DATA_DIR,
        'working_dir': base_working,
        'model_dir': os.path.join(base_working, 'models'),
        'output_dir': os.path.join(base_working, 'output'),
        'det_img_dir': os.path.join(base_working, 'det_images'),
        'rec_img_dir': os.path.join(base_working, 'rec_images'),
        'test_img_dir': os.path.join(base_working, 'test_images'),
    }

    # Expected file names inside the data directory
    paths['det_csv'] = os.path.join(DATA_DIR, 'license_plates_detection_train.csv')
    paths['rec_csv'] = os.path.join(DATA_DIR, 'license_plates_recognition_train.csv')
    paths['det_zip'] = os.path.join(DATA_DIR, 'license_plates_detection_train.zip')
    paths['rec_zip'] = os.path.join(DATA_DIR, 'license_plates_recognition_train.zip')
    paths['test_zip'] = os.path.join(DATA_DIR, 'test.zip')

    # Create writable directories
    for key, path in paths.items():
        if key not in ['det_csv', 'rec_csv', 'det_zip', 'rec_zip', 'test_zip', 'data_dir']:
            os.makedirs(path, exist_ok=True)

    return paths

## 2. DATA EXTRACTION UTILS

In [ ]:
def extract_zip_safe(zip_path, dest_dir):
    if not os.path.exists(zip_path):
        print(f"⚠️ Warning: Zip file not found at {zip_path}")
        return
    
    # Check if already extracted (if dest_dir has jpg files)
    if os.path.exists(dest_dir) and any(f.endswith('.jpg') for f in os.listdir(dest_dir)):
        print(f"✅ Skipping extraction: {dest_dir} already contains images.")
        return

    print(f"📦 Extracting {os.path.basename(zip_path)} to {dest_dir}...")
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(dest_dir)
        print("✅ Extraction complete.")
    except Exception as e:
        print(f"❌ Error extracting zip: {e}")

def find_image_root(base_dir):
    """Finds the deepest directory containing .jpg files."""
    if not os.path.exists(base_dir):
        return base_dir
    
    for root, dirs, files in os.walk(base_dir):
        if any(f.lower().endswith('.jpg') for f in files):
            return root
    return base_dir

## 3. HELPER FUNCTIONS

In [ ]:
def parse_plate_text(text: str) -> str:
    """
    Converts raw text (e.g., '94T3458') to fixed 7-digit string ('0943458').
    Handles missing 'T' or malformed inputs gracefully.
    """
    text = str(text).strip().upper()
    if 'T' in text:
        parts = text.split('T', 1)
        left = re.sub(r'\D', '', parts[0])
        right = re.sub(r'\D', '', parts[1])
        left = left.zfill(3)[:3]
        right = right.zfill(4)[:4]
        combined = left + right
    else:
        digits = re.sub(r'\D', '', text)
        combined = digits.zfill(7)[:7]
    
    # Final safety check
    return re.sub(r'\D', '0', combined).zfill(7)[:7]

## 4. DATASETS

In [ ]:
class DetectionDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # Robust column name handling
        img_name = row.get('img_id') or row.get('filename') or row.get('id') or str(idx) + '.jpg'
        
        img_path = os.path.join(self.image_dir, str(img_name))
        if not os.path.exists(img_path):
            # Fallback for nested folders if direct path fails
            for root, _, files in os.walk(self.image_dir):
                if img_name in files:
                    img_path = os.path.join(root, img_name)
                    break
        
        img = Image.open(img_path).convert('RGB')
        W, H = img.size
        
        # Normalize bbox [ymin, xmin, ymax, xmax] to 0-1
        ymin = float(row['ymin']) / H
        xmin = float(row['xmin']) / W
        ymax = float(row['ymax']) / H
        xmax = float(row['xmax']) / W
        
        bbox = torch.tensor([ymin, xmin, ymax, xmax], dtype=torch.float32)
        img = self.transform(img)
        
        return img, bbox, str(img_name)

class RecognitionDataset(Dataset):
    def __init__(self, csv_path, image_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_name = row.get('img_id') or row.get('filename') or row.get('id') or str(idx) + '.jpg'
        
        img_path = os.path.join(self.image_dir, str(img_name))
        if not os.path.exists(img_path):
             for root, _, files in os.walk(self.image_dir):
                if img_name in files:
                    img_path = os.path.join(root, img_name)
                    break

        img = Image.open(img_path).convert('RGB') # ResNet expects 3 channels
        
        # Use robust parsing
        digits_str = parse_plate_text(row['text'])
        labels = torch.tensor([int(d) for d in digits_str], dtype=torch.long)
        
        img = self.transform(img)
        return img, labels, str(img_name)

## 5. MODELS

In [ ]:
class LicensePlateDetector(nn.Module):
    """VGG16 based regressor for bounding box [ymin, xmin, ymax, xmax]"""
    def __init__(self, pretrained=True):
        super().__init__()
        vgg = models.vgg16(weights='IMAGENET1K_V1' if pretrained else None)
        self.backbone = vgg.features
        self.avgpool = vgg.avgpool
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512 * 7 * 7, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 4),
            nn.Sigmoid() # Output normalized coordinates
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.avgpool(x)
        return self.head(x)

class LicensePlateRecognizer(nn.Module):
    """ResNet18 based multi-head classifier for 7 digits"""
    def __init__(self, pretrained=True, num_digits=7, num_classes=10):
        super().__init__()
        resnet = models.resnet18(weights='IMAGENET1K_V1' if pretrained else None)
        feat_dim = resnet.fc.in_features
        resnet.fc = nn.Identity()
        self.backbone = resnet
        
        # 7 separate heads for each digit position
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(feat_dim, 128), 
                nn.ReLU(), 
                nn.Dropout(0.2),
                nn.Linear(128, num_classes)
            )
            for _ in range(num_digits)
        ])

    def forward(self, x):
        feats = self.backbone(x)
        # Stack outputs: (Batch, 7, 10)
        return torch.stack([h(feats) for h in self.heads], dim=1)

## 6. TRAINING LOGIC

In [ ]:
def train_detector(paths, device, epochs=30, batch_size=32):
    print("\n" + "="*50)
    print("TRAINING LICENSE PLATE DETECTOR")
    print("="*50)
    
    det_tfm_val = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    det_tfm_train = T.Compose([
        T.Resize((224, 224)),
        T.ColorJitter(brightness=0.2, contrast=0.2),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_ds = pd.read_csv(paths['det_csv'])
    indices = np.arange(len(full_ds))
    np.random.seed(42)
    np.random.shuffle(indices)
    val_split = int(len(full_ds) * 0.1)
    train_indices, val_indices = indices[val_split:], indices[:val_split]

    train_ds = DetectionDataset(paths['det_csv'], paths['det_img_dir'], transform=det_tfm_train)
    train_ds.df = train_ds.df.iloc[train_indices].reset_index(drop=True)
    
    val_ds = DetectionDataset(paths['det_csv'], paths['det_img_dir'], transform=det_tfm_val)
    val_ds.df = val_ds.df.iloc[val_indices].reset_index(drop=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = LicensePlateDetector(pretrained=True).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.SmoothL1Loss()

    best_loss = float('inf')

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Det Epoch {epoch}/{epochs}', leave=False)
        
        for imgs, bboxes, _ in pbar:
            imgs, bboxes = imgs.to(device), bboxes.to(device)
            optimizer.zero_grad()
            preds = model(imgs)
            loss = criterion(preds, bboxes)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        scheduler.step()
        avg_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, bboxes, _ in val_loader:
                imgs, bboxes = imgs.to(device), bboxes.to(device)
                preds = model(imgs)
                val_loss += criterion(preds, bboxes).item()
        val_loss /= len(val_loader)

        print(f"Epoch {epoch}: Train Loss {avg_loss:.4f} | Val Loss {val_loss:.4f}")

        if val_loss < best_loss:
            best_loss = val_loss
            torch.save(model.state_dict(), os.path.join(paths['model_dir'], 'best_detector.pth'))
            print(f"💾 Saved best detector model (Loss: {best_loss:.4f})")

    return model

def train_recognizer(paths, device, epochs=40, batch_size=32):
    print("\n" + "="*50)
    print("TRAINING LICENSE PLATE RECOGNIZER")
    print("="*50)
    
    rec_tfm_val = T.Compose([
        T.Resize((64, 200)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    rec_tfm_train = T.Compose([
        T.Resize((64, 200)),
        T.ColorJitter(brightness=0.3, contrast=0.3),
        T.RandomAffine(degrees=2, translate=(0.05, 0.05)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_ds = pd.read_csv(paths['rec_csv'])
    indices = np.arange(len(full_ds))
    np.random.seed(42)
    np.random.shuffle(indices)
    val_split = int(len(full_ds) * 0.1)
    train_indices, val_indices = indices[val_split:], indices[:val_split]

    train_ds = RecognitionDataset(paths['rec_csv'], paths['rec_img_dir'], transform=rec_tfm_train)
    train_ds.df = train_ds.df.iloc[train_indices].reset_index(drop=True)
    
    val_ds = RecognitionDataset(paths['rec_csv'], paths['rec_img_dir'], transform=rec_tfm_val)
    val_ds.df = val_ds.df.iloc[val_indices].reset_index(drop=True)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    model = LicensePlateRecognizer(pretrained=True).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_acc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        pbar = tqdm(train_loader, desc=f'Rec Epoch {epoch}/{epochs}', leave=False)
        
        for imgs, labels, _ in pbar:
            imgs, labels = imgs.to(device), labels.to(device) # labels shape: (B, 7)
            optimizer.zero_grad()
            
            logits = model(imgs) # Shape: (B, 7, 10)
            
            # Calculate loss for each of the 7 digit positions
            loss = 0
            for d in range(7):
                loss += criterion(logits[:, d, :], labels[:, d])
            
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        scheduler.step()
        avg_loss = running_loss / len(train_loader)

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                preds = logits.argmax(dim=2) # (B, 7)
                correct += (preds == labels).sum().item()
                total += labels.numel()
        
        acc = correct / total
        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), os.path.join(paths['model_dir'], 'best_recognizer.pth'))
            print(f"💾 Saved best recognizer model (Acc: {acc:.4f})")

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch}: Train Loss {avg_loss:.4f} | Val Acc {acc:.4f} | Best Acc {best_acc:.4f}")

    return model

## 7. INFERENCE & SUBMISSION

In [ ]:
def generate_submission(paths, device):
    print("\n" + "="*50)
    print("GENERATING SUBMISSION FILE")
    print("="*50)
    
    # Load Models
    det_model = LicensePlateDetector(pretrained=False).to(device)
    det_model.load_state_dict(torch.load(os.path.join(paths['model_dir'], 'best_detector.pth'), map_location=device))
    det_model.eval()

    rec_model = LicensePlateRecognizer(pretrained=False).to(device)
    rec_model.load_state_dict(torch.load(os.path.join(paths['model_dir'], 'best_recognizer.pth'), map_location=device))
    rec_model.eval()

    # Transforms
    det_tfm = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    rec_tfm = T.Compose([
        T.Resize((64, 200)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    test_files = sorted([f for f in os.listdir(paths['test_img_dir']) if f.lower().endswith('.jpg')])
    print(f"Found {len(test_files)} test images.")

    rows = []
    
    pbar = tqdm(test_files, desc="Inference")
    for fname in pbar:
        img_id = os.path.splitext(fname)[0]
        img_path = os.path.join(paths['test_img_dir'], fname)
        
        try:
            full_img = Image.open(img_path).convert('RGB')
            W, H = full_img.size
            
            # 1. Detect
            t_det = det_tfm(full_img).unsqueeze(0).to(device)
            with torch.no_grad():
                bbox_norm = det_model(t_det)[0].cpu().numpy()
            
            y1, x1, y2, x2 = int(bbox_norm[0]*H), int(bbox_norm[1]*W), int(bbox_norm[2]*H), int(bbox_norm[3]*W)
            
            # Clamp coordinates
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            
            if x2 <= x1 or y2 <= y1:
                raise ValueError("Invalid bounding box")

            crop = full_img.crop((x1, y1, x2, y2))

            # 2. Recognize
            t_rec = rec_tfm(crop.convert('RGB')).unsqueeze(0).to(device)
            with torch.no_grad():
                logits = rec_model(t_rec)[0]
            probs = torch.softmax(logits, dim=1).cpu().numpy() # (7, 10)
            
        except Exception as e:
            # Fallback: uniform probability if detection fails
            probs = np.ones((7, 10)) / 10.0
        
        # 3. Format Rows (7 rows per image)
        for d in range(7):
            row_id = f"{img_id}_{d+1}"
            # Format probabilities to 6 decimal places
            row_data = [row_id] + [f"{p:.6f}" for p in probs[d]]
            rows.append(row_data)

    # Create DataFrame
    cols = ['id'] + [str(i) for i in range(10)]
    sub_df = pd.DataFrame(rows, columns=cols)
    
    out_path = os.path.join(paths['output_dir'], 'submission.csv')
    sub_df.to_csv(out_path, index=False)
    print(f"\n✅ Submission saved to: {out_path}")
    print(f"Shape: {sub_df.shape}")
    print(sub_df.head(7))
    
    return out_path

## 8. MAIN EXECUTION

In [ ]:
if __name__ == "__main__":
    # Setup
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using Device: {DEVICE}")
    
    try:
        paths = setup_paths()
        
        # Extract Data
        extract_zip_safe(paths['det_zip'], paths['det_img_dir'])
        extract_zip_safe(paths['rec_zip'], paths['rec_img_dir'])
        extract_zip_safe(paths['test_zip'], paths['test_img_dir'])
        
        # Update roots in case of nested folders
        paths['det_img_dir'] = find_image_root(paths['det_img_dir'])
        paths['rec_img_dir'] = find_image_root(paths['rec_img_dir'])
        paths['test_img_dir'] = find_image_root(paths['test_img_dir'])
        
        print(f"Detection Images Root: {paths['det_img_dir']}")
        print(f"Recognition Images Root: {paths['rec_img_dir']}")
        print(f"Test Images Root: {paths['test_img_dir']}")

        # Check if data exists before training
        if not os.path.exists(paths['det_csv']) or not os.path.exists(paths['rec_csv']):
            print("❌ Critical Error: CSV files not found. Ensure dataset is uploaded correctly.")
        else:
            # Train
            # Note: In a real Kaggle kernel with time limits, you might load pre-trained weights 
            # instead of training from scratch every time. Here we train for completeness.
            train_detector(paths, DEVICE, epochs=15) # Reduced epochs for demo/speed, increase to 30 for full score
            train_recognizer(paths, DEVICE, epochs=20) # Reduced epochs for demo/speed, increase to 40 for full score
            
            # Infer
            generate_submission(paths, DEVICE)
            
    except Exception as e:
        print(f"❌ An error occurred: {e}")
        import traceback
        traceback.print_exc()